# Tesseract OCR on PDF

Runs Tesseract OCR over each page of a PDF using the same stack as `ocr_scrub.py`:
`pypdfium2` to rasterize pages, `pytesseract` to OCR.

Prereq: the `tesseract` binary must be installed (on macOS: `brew install tesseract`).

In [1]:
import pypdfium2 as pdfium
import pytesseract
from pathlib import Path

In [2]:
PDF_PATH = Path("/Users/peterchou/Downloads/fake_w2.pdf")  # <- change to your PDF
DPI = 300
SCALE = DPI / 72.0  # points → pixels, matches ocr_scrub.py

In [3]:
pdf = pdfium.PdfDocument(str(PDF_PATH))
print(f"{PDF_PATH} has {len(pdf)} page(s)")

/Users/peterchou/Downloads/fake_w2.pdf has 1 page(s)


In [4]:
page_texts: list[str] = []
for page_idx in range(len(pdf)):
    page = pdf[page_idx]
    bitmap = page.render(scale=SCALE)
    img = bitmap.to_pil()
    text = pytesseract.image_to_string(img)
    page_texts.append(text)
    print(f"\n===== Page {page_idx + 1} =====")
    print(text)


===== Page 1 =====
Sample W-2

To read a brief description of a box on the W-2, move your mouse pointer over a box in the sample form.

Important: The information in this sample is specific to RUN powered by ADP only and should not be used as the official W-2. In addition
to this sample, it's important that you use the detailed instructions for Form W-2 found here: https://www.irs.gov/pub/i

NOTE: New for Tax Year 2020, marital status and exemptions have been removed based on the 2020 W-4 changes.

W-2

Employee Reference
Wage and Tax

Copy C for employee's records.

d

Control number | Dept.

000011 R#/123

c

Corp.

Copy

OMB No. 1545-0008
Employer use only
22

Employer’s name, address, and ZIP code

SAMPLE COMPANY INC

123 MAIN ST

ANYWHERE, CA 123456 1234

2020 W-2 and EARNINGS SUMMARY 22>

This section displays the breakdown of your total gross pay during the tax year and any adjustments
to determine the “Reported W-2 Wages”. These amounts should match your final pay statement of

In [5]:
# Word-level data (bbox + confidence) for page 1, same call as ocr_scrub.py
page = pdf[0]
img = page.render(scale=SCALE).to_pil()
data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
for i, word in enumerate(data["text"]):
    if not word.strip():
        continue
    print(f"{word!r:30} conf={data['conf'][i]} bbox=(left={data['left'][i]}, top={data['top'][i]}, w={data['width'][i]}, h={data['height'][i]})")

'Sample'                       conf=91 bbox=(left=104, top=49, w=216, h=64)
'W-2'                          conf=91 bbox=(left=336, top=49, w=120, h=51)
'To'                           conf=95 bbox=(left=104, top=133, w=50, h=41)
'read'                         conf=95 bbox=(left=169, top=133, w=94, h=41)
'a'                            conf=95 bbox=(left=279, top=146, w=21, h=28)
'brief'                        conf=95 bbox=(left=317, top=133, w=107, h=41)
'description'                  conf=95 bbox=(left=436, top=133, w=252, h=51)
'of'                           conf=95 bbox=(left=703, top=133, w=47, h=41)
'a'                            conf=96 bbox=(left=762, top=146, w=22, h=28)
'box'                          conf=96 bbox=(left=800, top=133, w=77, h=41)
'on'                           conf=96 bbox=(left=891, top=146, w=51, h=28)
'the'                          conf=92 bbox=(left=957, top=133, w=73, h=41)
'W-2,'                         conf=92 bbox=(left=1044, top=133, w=105, h=48)
'move'  

In [ ]:
out_path = PDF_PATH.with_suffix(".txt")
out_path.write_text("\n\n".join(page_texts))
print(f"Wrote {out_path}")

In [ ]:
pdf.close()

In [3]:
a = """
[Page 1]
Booking.com .
Confirmed
Your booking details
Customer reference: 40-913153737
PIN code: 3521
Tokyo (HND) to Guangzhou (CAN)
Sun, Apr 12 : 3:40 PM - Sun, Apr 12 : 7:15 PM
Booking reference:
Direct : 4h 35m : Economy
QGEKHW
China Southern Airlines : CZ386
Traveler details
Peng Zhou
Adult - Male : Sep 10 1991
Travel document details
EJ4963642 : US : 2031-07-26
Baggage
Total number of bags included for all travelers
Flight to Guangzhou
1 personal item
Fits under the seat in front of you .
1 carry-on bag .
7.9 x 15.8 x 21.7 inches. Max weight 17.6 Ibs

[Page 2]
Booking.com
2 checked bags
Max weight 50.7 Ibs
Seats
Flight to Guangzhou
: 1 seat selected
35D
Boarding and flight
Meal choice
Gluten-free (x1)
Payment
Total
USD 412.18
Includes taxes and fees
Contact details
+17813331331
danchou9183@gmail.com
"""

len(a)


817